# 03d — QAOA on AWS Braket SV1 (Trial Selection QUBO)

This notebook runs QAOA against the Scenario B trial-selection QUBO produced in `03b`.

Inputs (from repo artifacts):
- `data/qubo/scenario_B_qubo.json`
- `data/qubo/scenario_B_qubo_metadata.csv`
- Classical baselines (optional, for comparison):
  - `data/scenarios/scenario_B_greedy_selection.csv`
  - `data/qubo/scenario_B_qubo_sa_solution.csv`

Outputs (new artifacts):
- `data/results/03d_sv1_best_bitstring.txt`
- `data/results/03d_sv1_summary.csv`
- `data/results/03d_sv1_selected_trials.csv`

Notes:
- We restrict to a manageable number of variables (K) for statevector-based expectation evaluation.
- We compute expectation values from the statevector (shots=0) for stable optimization.


In [1]:
# ============================================================
# Cell 1 — Load QUBO + metadata; optionally reduce to K vars
# ============================================================

from pathlib import Path
import json
import numpy as np
import pandas as pd

def log(msg: str) -> None:
    print(msg)

QUBO_PATH = Path("data/qubo/scenario_B_qubo.json")
META_PATH = Path("data/qubo/scenario_B_qubo_metadata.csv")

if not QUBO_PATH.exists():
    raise FileNotFoundError(f"Missing {QUBO_PATH}. Run 03b first.")
if not META_PATH.exists():
    raise FileNotFoundError(f"Missing {META_PATH}. Run 03b first.")

# Load Q (string "i,j" -> tuple(i,j))
with QUBO_PATH.open("r") as f:
    Q_serializable = json.load(f)

Q_full = {}
for key, val in Q_serializable.items():
    i_str, j_str = key.split(",")
    Q_full[(int(i_str), int(j_str))] = float(val)

meta_full = pd.read_csv(META_PATH)

if "var_index" not in meta_full.columns:
    raise ValueError("Metadata missing 'var_index'.")

N_full = int(meta_full["var_index"].nunique())
log(f"[Cell 1] Loaded full QUBO: N={N_full}, nnz={len(Q_full)}. Metadata shape={meta_full.shape}")

# --- Choose a manageable subset size K (statevector scales as 2^K) ----------
# Practical: K=16..20 (2^20=1,048,576 amplitudes) depending on machine/time.
K = min(18, N_full)

# Choose "best" variables for a smaller QAOA run:
# heuristic: highest benefit per cost, with safe guards for zeros.
meta = meta_full.copy()

if "benefit_score" not in meta.columns:
    raise ValueError("Metadata missing 'benefit_score' (expected from Phase 2/02c outputs).")
if "estimated_trial_cost" not in meta.columns:
    raise ValueError("Metadata missing 'estimated_trial_cost'.")

meta["cost_safe"] = meta["estimated_trial_cost"].replace(0, np.nan)
meta["benefit_per_cost"] = meta["benefit_score"] / meta["cost_safe"]
meta["benefit_per_cost"] = meta["benefit_per_cost"].fillna(0.0)

# Pick top K by benefit_per_cost (tie-breaker: benefit_score)
meta_top = (
    meta.sort_values(["benefit_per_cost", "benefit_score"], ascending=False)
        .head(K)
        .copy()
)

selected_var_indices = meta_top["var_index"].tolist()

# Re-index variables to 0..K-1 for this subproblem
old_to_new = {old: new for new, old in enumerate(selected_var_indices)}
new_to_old = {v: k for k, v in old_to_new.items()}

# Restrict QUBO to selected variables
Q = {}
for (i, j), coeff in Q_full.items():
    if i in old_to_new and j in old_to_new:
        Q[(old_to_new[i], old_to_new[j])] = coeff

# Sub-metadata aligned to new indices
meta_sub = meta_top.copy()
meta_sub["var_index_old"] = meta_sub["var_index"]
meta_sub["var_index"] = meta_sub["var_index_old"].map(old_to_new)
meta_sub = meta_sub.sort_values("var_index").reset_index(drop=True)

N = K
log(f"[Cell 1] Subproblem: K={K}, Q nnz={len(Q)}, meta_sub shape={meta_sub.shape}")
meta_sub.head()


[Cell 1] Loaded full QUBO: N=40, nnz=820. Metadata shape=(40, 14)
[Cell 1] Subproblem: K=18, Q nnz=171, meta_sub shape=(18, 17)


,var_index,nct_id,brief_title,overall_status,phase,conditions,interventions,location_countries,lead_sponsor,lead_sponsor_norm,region_label,estimated_trial_cost,enrollment_feasibility_score,benefit_score,cost_safe,benefit_per_cost,var_index_old
0,0,NCT00003042,Chemotherapy and Stem Cell Transplantation in ...,"Active, not recruiting",Phase 2,['Breast Cancer'],['filgrastim' 'cisplatin' 'cyclophosphamide' '...,['United States'],City of Hope Medical Center,City of Hope Medical Center,Global / Multi-Region,2.0,1.0,0.6,2.0,0.3,0
1,1,NCT06234631,Cannabidiol for Postoperative Opioid Reduction...,Recruiting,Phase 2,"['Knee Replacement Surgery' 'Osteoarthritis, K...",['Epidiolex oral solution' 'Placebo'],['United States'],Chad Brummett,Chad Brummett,Global / Multi-Region,2.0,1.0,0.6,2.0,0.3,1
2,2,NCT06257537,Sustained Acoustic Medicine for Symptomatic Tr...,Recruiting,Phase 2,['Osteo Arthritis Knee' 'Arthritis'],['Sustained Acoustic Device with 2.5% Diclofen...,['United States'],"ZetrOZ, Inc.","ZetrOZ, Inc.",Global / Multi-Region,2.0,1.0,0.6,2.0,0.3,2
3,3,NCT06257875,A Study to Assess Adverse Events and Change in...,"Active, not recruiting",Phase 2,['Ulcerative Colitis'],['Lutikizumab' 'Lutikizumab' 'Adalimumab'],['Australia' 'Austria' 'Belgium' 'Bulgaria' 'C...,AbbVie,AbbVie,Global / Multi-Region,2.0,1.0,0.6,2.0,0.3,3
4,4,NCT06259123,Neoadjuvant PSMA-RLT in Oligometastatic PCa,Recruiting,Phase 2,['Prostate Cancer'],['[177Lu]Lu-PSMA I&T'],['Austria'],Medical University of Vienna,Medical University of Vienna,Global / Multi-Region,2.0,1.0,0.6,2.0,0.3,4


### What Cell 1 Just Did

This cell loaded the Scenario B QUBO (`data/qubo/scenario_B_qubo.json`) and its variable-level metadata (`data/qubo/scenario_B_qubo_metadata.csv`) to reconstruct the full optimization problem (number of variables **N** and nonzero QUBO terms).

Because statevector-based QAOA scales exponentially with the number of variables (2^K), the cell then created a **manageable subproblem** by selecting the top **K** candidate variables using a simple **benefit-per-cost** heuristic (with safeguards for zero/empty costs). Finally, it **re-indexed** the chosen variables to a compact 0..K−1 range, **restricted** the QUBO dictionary to those variables only, and produced an aligned `meta_sub` table so any selected bitstring can be mapped back to real trial records.


In [2]:
# ============================================================
# Cell 2 — Convert QUBO (x in {0,1}) to Ising (Z eigenvalues)
# ============================================================

from collections import defaultdict

def qubo_to_ising(Q: dict, N: int):
    """
    Convert QUBO over x in {0,1} to Ising over z in {+1,-1} using:
      x_i = (1 - z_i) / 2

    Returns:
      constant: float
      h: dict i -> float        (Z_i coefficient)
      J: dict (i,j) -> float    (Z_i Z_j coefficient), with i<j
    """
    constant = 0.0
    h = defaultdict(float)
    J = defaultdict(float)

    for (i, j), q in Q.items():
        if i == j:
            # q * x_i = q*(1 - z_i)/2 = q/2 - (q/2) z_i
            constant += q / 2.0
            h[i] += -q / 2.0
        else:
            # q * x_i x_j
            # = q * (1/4 - z_i/4 - z_j/4 + z_i z_j/4)
            constant += q / 4.0
            h[i] += -q / 4.0
            h[j] += -q / 4.0
            a, b = (i, j) if i < j else (j, i)
            J[(a, b)] += q / 4.0

    return float(constant), dict(h), dict(J)

const_E, h, J = qubo_to_ising(Q, N)

log(f"[Cell 2] Ising conversion complete.")
log(f"[Cell 2] constant offset: {const_E:.6f}")
log(f"[Cell 2] linear terms |h|:  {len(h)}")
log(f"[Cell 2] ZZ terms     |J|:  {len(J)}")

# Small peek
list(h.items())[:5], list(J.items())[:5]


[Cell 2] Ising conversion complete.
[Cell 2] constant offset: 534.600000
[Cell 2] linear terms |h|:  18
[Cell 2] ZZ terms     |J|:  153


([(0, -199.7), (1, -199.7), (2, -199.7), (3, -199.7), (4, -199.7)],
 [((0, 1), 20.0),
  ((0, 2), 20.0),
  ((0, 3), 20.0),
  ((0, 4), 20.0),
  ((0, 5), 20.0)])

### What Cell 2 Just Did

This cell converted the reduced QUBO objective (binary variables `x ∈ {0,1}`) into an equivalent **Ising form**
expressed with Pauli-Z eigenvalues `z ∈ {+1,−1}` using the standard mapping:

`x_i = (1 − z_i) / 2`

The result is:
- a constant offset (which does not affect optimization decisions),
- linear Z coefficients `h_i`, and
- quadratic ZZ couplings `J_ij`.

These coefficients let us build the QAOA cost unitary using only single-qubit `Rz` rotations and two-qubit ZZ phase
constructions implemented via `CNOT–Rz–CNOT`.


In [3]:
# ============================================================
# Cell 3 — QAOA circuit builder + drivers (Local + SV1)
#   (FIXED: Braket task results must go to an amazon-braket-* bucket)
# ============================================================

import math
import numpy as np

from braket.circuits import Circuit, ResultType
from braket.aws import AwsDevice, AwsSession
from braket.devices import Devices
from braket.devices import LocalSimulator

# --- Braket output destination (SV1 jobs write results to S3) ----------------
# IMPORTANT:
# Braket CreateQuantumTask requires the results bucket to be Braket-managed
# (typically starts with "amazon-braket-"). The easiest reliable option is to use
# AwsSession().default_bucket().

aws_sess = AwsSession()
BRAKET_RESULTS_BUCKET = aws_sess.default_bucket()
BRAKET_RESULTS_PREFIX = "phase3/qaoa_trial_selection"  # choose any prefix you like

# Cost control:
# - Optimize on LocalSimulator (fast + free)
# - Do a single final confirmation run on SV1
OPTIMIZE_LOCALLY = True
FINAL_RUN_ON_SV1 = True

log(f"[Cell 3] Braket results bucket: {BRAKET_RESULTS_BUCKET}")
log(f"[Cell 3] Braket results prefix: {BRAKET_RESULTS_PREFIX}")
log(f"[Cell 3] OPTIMIZE_LOCALLY={OPTIMIZE_LOCALLY}, FINAL_RUN_ON_SV1={FINAL_RUN_ON_SV1}")


def build_qaoa_circuit(N: int, h: dict, J: dict, gammas: np.ndarray, betas: np.ndarray) -> Circuit:
    """
    Build a QAOA circuit for Ising Hamiltonian:
      H = sum_i h_i Z_i + sum_{i<j} J_ij Z_i Z_j

    We use:
      - Rz for linear Z terms
      - CNOT-Rz-CNOT for ZZ terms
      - Rx as the mixer
    """
    p = len(gammas)
    if len(betas) != p:
        raise ValueError("gammas and betas must have the same length (p).")

    circ = Circuit()

    # Start in |+>^N
    for q in range(N):
        circ.h(q)

    for layer in range(p):
        gamma = float(gammas[layer])
        beta = float(betas[layer])

        # Cost unitary: exp(-i gamma * H)
        for i, hi in h.items():
            theta = 2.0 * gamma * hi
            if theta != 0.0:
                circ.rz(i, theta)

        for (i, j), Jij in J.items():
            theta = 2.0 * gamma * Jij
            if theta != 0.0:
                circ.cnot(i, j)
                circ.rz(j, theta)
                circ.cnot(i, j)

        # Mixer: exp(-i beta * sum X_i) -> Rx(2*beta)
        for q in range(N):
            circ.rx(q, 2.0 * beta)

    # Deterministic evaluation
    circ.add_result_type(ResultType.StateVector())
    return circ


def statevector_from_task_result(result):
    """
    Extract statevector from Braket result.
    For a Circuit with ResultType.StateVector(), result.values[0] should be the SV.
    """
    if hasattr(result, "values") and len(result.values) > 0:
        return np.asarray(result.values[0], dtype=np.complex128)
    raise RuntimeError("Unable to extract statevector from Braket result (no .values found).")


def qubo_cost_for_bitstrings(Q: dict, bitstrings: np.ndarray) -> np.ndarray:
    """
    Compute QUBO cost for many bitstrings.
    bitstrings: array shape (M, N) with 0/1 entries where column i is variable/qubit i.
    """
    M, N = bitstrings.shape
    costs = np.zeros(M, dtype=np.float64)

    for (i, j), q in Q.items():
        if i == j:
            costs += q * bitstrings[:, i]
        else:
            costs += q * bitstrings[:, i] * bitstrings[:, j]

    return costs


def enumerate_all_bitstrings_qubit0_first(N: int) -> np.ndarray:
    """
    Enumerate all bitstrings of length N as a matrix (2^N x N),
    where column i corresponds to qubit/variable i.

    We use a *qubit0-first* convention here:
      bit[i] = (basis_index >> i) & 1
    This keeps the mapping consistent when decoding selections as "bit i -> variable i".
    """
    M = 2 ** N
    idx = np.arange(M, dtype=np.uint32)
    bits = ((idx[:, None] >> np.arange(N)) & 1).astype(np.int8)  # col i is qubit i
    return bits


def expected_qubo_cost_from_statevector(Q: dict, statevector: np.ndarray, N: int) -> float:
    """
    Compute E[C(x)] exactly from statevector probabilities, using the same
    *qubit0-first* bit convention as enumerate_all_bitstrings_qubit0_first().
    """
    probs = np.abs(statevector) ** 2
    M = 2 ** N
    if probs.shape[0] != M:
        raise ValueError(f"statevector length {probs.shape[0]} != 2^N={M}")

    bits = enumerate_all_bitstrings_qubit0_first(N)
    costs = qubo_cost_for_bitstrings(Q, bits)
    return float(np.sum(probs * costs))


def bitstring_from_bits_row(bits_row: np.ndarray) -> str:
    """
    Convert a bits row (length N, bits_row[i] is bit for qubit i) to a string,
    written left-to-right as qubit0..qubitN-1.
    """
    return "".join(str(int(b)) for b in bits_row.tolist())


def run_statevector_local(circuit: Circuit):
    dev = LocalSimulator()
    task = dev.run(circuit, shots=0)
    return task.result()


def run_statevector_sv1(circuit: Circuit):
    device = AwsDevice(Devices.Amazon.SV1, aws_session=aws_sess)
    task = device.run(
        circuit,
        s3_destination_folder=(BRAKET_RESULTS_BUCKET, BRAKET_RESULTS_PREFIX),
        shots=0,
    )
    return task.result()


def objective(params: np.ndarray, p: int, use_sv1: bool) -> float:
    """
    params = [gamma_0..gamma_{p-1}, beta_0..beta_{p-1}]
    Returns expected QUBO cost (minimize).
    """
    gammas = params[:p]
    betas = params[p:]

    circ = build_qaoa_circuit(N, h, J, gammas, betas)

    res = run_statevector_sv1(circ) if use_sv1 else run_statevector_local(circ)
    sv = statevector_from_task_result(res)
    return expected_qubo_cost_from_statevector(Q, sv, N)


[Cell 3] Braket results bucket: amazon-braket-us-west-2-581610642254
[Cell 3] Braket results prefix: phase3/qaoa_trial_selection
[Cell 3] OPTIMIZE_LOCALLY=True, FINAL_RUN_ON_SV1=True


### What Cell 3 Just Did

This cell set up the **execution layer** for QAOA:

- Built a QAOA circuit constructor that:
  - initializes `|+⟩^N`,
  - applies `p` repeating layers of:
    - a cost unitary derived from `(h, J)` using `Rz` and `CNOT–Rz–CNOT`,
    - an `Rx` mixer on every qubit,
  - requests a **StateVector** output so expectation values can be computed exactly.

- Implemented deterministic evaluation of the objective by:
  - extracting the statevector,
  - enumerating all `2^N` bitstrings where **bit i maps to qubit i**, and
  - computing the expected QUBO cost `E[C(x)]` from state probabilities.

- Added two execution modes:
  - **LocalSimulator** for fast, inexpensive optimization,
  - **SV1** for an AWS confirmation run and artifact capture.

In [4]:
# ============================================================
# Cell 4 — Optimize QAOA parameters (SciPy if available)
# ============================================================

p = 1  # Start simple; raise to 2/3 once the pipeline is stable

# Initial angles (reasonable small values)
x0 = np.zeros(2 * p, dtype=np.float64)
x0[:p] = 0.3  # gammas
x0[p:] = 0.7  # betas

use_sv1_for_objective = (not OPTIMIZE_LOCALLY)

log(f"[Cell 4] Starting optimization: p={p}, N={N}, objective_on={'SV1' if use_sv1_for_objective else 'LocalSimulator'}")

best_params = None
best_obj = None

try:
    from scipy.optimize import minimize

    res = minimize(
        fun=lambda x: objective(x, p, use_sv1=use_sv1_for_objective),
        x0=x0,
        method="COBYLA",
        options={"maxiter": 30, "disp": True},
    )

    best_params = res.x
    best_obj = float(res.fun)

    log(f"[Cell 4] Optimization complete.")
    log(f"[Cell 4] best expected QUBO cost: {best_obj:.6f}")
    log(f"[Cell 4] best params: {best_params}")

except Exception as e:
    log(f"[Cell 4] SciPy minimize unavailable/failed ({e}). Using random search fallback.")

    rng = np.random.default_rng(42)
    trials = 20

    best_obj = float("inf")
    best_params = None

    for t in range(trials):
        gammas = rng.uniform(0.0, math.pi, size=p)
        betas = rng.uniform(0.0, math.pi / 2.0, size=p)
        params = np.concatenate([gammas, betas])

        val = objective(params, p, use_sv1=use_sv1_for_objective)
        log(f"[Cell 4] Trial {t+1}/{trials}: expected cost={val:.6f}")

        if val < best_obj:
            best_obj = val
            best_params = params

    log(f"[Cell 4] Random search complete.")
    log(f"[Cell 4] best expected QUBO cost: {best_obj:.6f}")
    log(f"[Cell 4] best params: {best_params}")


[Cell 4] Starting optimization: p=1, N=18, objective_on=LocalSimulator
Return from COBYLA because the objective function has been evaluated MAXFUN times.
Number of function values = 30   Least value of F = 534.5678539478833
The corresponding X is: [1.1357522 1.5972025]

[Cell 4] Optimization complete.
[Cell 4] best expected QUBO cost: 534.567854
[Cell 4] best params: [1.1357522 1.5972025]


### What Cell 4 Just Did

This cell performed a parameter search over QAOA angles `(γ, β)` to reduce the expected QUBO objective value.

- Primary path: SciPy `minimize()` with a gradient-free optimizer (COBYLA) for a bounded-evaluation search.
- Fallback path: a small random search if SciPy is not available.

By default, optimization happens on the **LocalSimulator** to keep iteration costs low. The best parameters found here
can then be validated with a single run on **SV1** to produce cloud-backed artifacts.


In [5]:
# ============================================================
# Cell 5 — Final run (SV1), decode bitstrings, and compare to baselines
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

def best_prob_and_best_cost_bitstrings(Q: dict, statevector: np.ndarray, N: int):
    """
    Returns:
      - most probable basis bitstring and its QUBO cost
      - minimum-cost basis bitstring (over all basis states) and its probability
    Bitstring convention: qubit0-first (position i -> qubit/variable i).
    """
    probs = np.abs(statevector) ** 2
    bits = enumerate_all_bitstrings_qubit0_first(N)  # from Cell 3
    costs = qubo_cost_for_bitstrings(Q, bits)        # from Cell 3

    k_prob = int(np.argmax(probs))
    k_cost = int(np.argmin(costs))

    bs_prob = bitstring_from_bits_row(bits[k_prob])  # from Cell 3
    bs_cost = bitstring_from_bits_row(bits[k_cost])

    return {
        "most_prob_bitstring": bs_prob,
        "most_prob_probability": float(probs[k_prob]),
        "most_prob_cost": float(costs[k_prob]),
        "min_cost_bitstring": bs_cost,
        "min_cost_probability": float(probs[k_cost]),
        "min_cost_cost": float(costs[k_cost]),
    }


def brute_force_global_optimum(Q: dict, N: int, max_bruteforce_N: int = 22):
    """
    Brute-force exact global optimum for the reduced subproblem.
    Guarded because 2^N grows quickly.
    """
    if N > max_bruteforce_N:
        return None, None

    bits = enumerate_all_bitstrings_qubit0_first(N)
    costs = qubo_cost_for_bitstrings(Q, bits)
    k = int(np.argmin(costs))
    return bitstring_from_bits_row(bits[k]), float(costs[k])


# --- Build final circuit with best angles found in Cell 4 --------------------
gammas = best_params[:p]
betas = best_params[p:]
circ_best = build_qaoa_circuit(N, h, J, gammas, betas)

# --- Run final (SV1 preferred) ----------------------------------------------
use_sv1_final = FINAL_RUN_ON_SV1
log(f"[Cell 5] Final run on {'SV1' if use_sv1_final else 'LocalSimulator'}...")

try:
    res_final = run_statevector_sv1(circ_best) if use_sv1_final else run_statevector_local(circ_best)
except Exception as e:
    log(f"[Cell 5] SV1 run failed ({e}). Falling back to LocalSimulator for final decode.")
    res_final = run_statevector_local(circ_best)

sv_final = statevector_from_task_result(res_final)

# Expected objective value (deterministic)
exp_cost_final = expected_qubo_cost_from_statevector(Q, sv_final, N)

# Decode bitstrings
decoded = best_prob_and_best_cost_bitstrings(Q, sv_final, N)

# Exact optimum baseline (if N small enough)
global_opt_bitstring, global_opt_cost = brute_force_global_optimum(Q, N, max_bruteforce_N=22)

log(f"[Cell 5] Expected QUBO cost (final): {exp_cost_final:.6f}")
log(f"[Cell 5] Most probable bitstring:   {decoded['most_prob_bitstring']}  "
    f"(p={decoded['most_prob_probability']:.3e}, cost={decoded['most_prob_cost']:.6f})")
log(f"[Cell 5] Min-cost basis bitstring:  {decoded['min_cost_bitstring']}   "
    f"(p={decoded['min_cost_probability']:.3e}, cost={decoded['min_cost_cost']:.6f})")

if global_opt_bitstring is not None:
    log(f"[Cell 5] Exact global optimum:      {global_opt_bitstring} (cost={global_opt_cost:.6f})")
else:
    log(f"[Cell 5] Exact global optimum:      skipped (N={N} too large for brute-force guard)")

# --- Map selection back to trial metadata -----------------------------------
# Convention: bit i corresponds to variable/qubit i
min_cost_bs = decoded["min_cost_bitstring"]
selected_positions = [i for i, ch in enumerate(min_cost_bs) if ch == "1"]

selected_trials = meta_sub[meta_sub["var_index"].isin(selected_positions)].copy()

# Robust sums (in case columns differ slightly across versions)
total_cost = float(selected_trials["estimated_trial_cost"].sum()) if "estimated_trial_cost" in selected_trials.columns else 0.0
total_benefit = float(selected_trials["benefit_score"].sum()) if "benefit_score" in selected_trials.columns else 0.0

log(f"[Cell 5] Min-cost bitstring selects {len(selected_trials)} trials (within K={N}).")
log(f"[Cell 5] Total estimated cost: {total_cost:,.2f}")
log(f"[Cell 5] Total benefit score:  {total_benefit:.6f}")

cols_to_show = [c for c in [
    "var_index", "nct_id", "phase", "overall_status",
    "estimated_trial_cost", "benefit_score", "lead_sponsor_norm", "region_label"
] if c in selected_trials.columns]

display(
    selected_trials[cols_to_show]
      .sort_values("benefit_score", ascending=False) if "benefit_score" in selected_trials.columns
      else selected_trials[cols_to_show]
)

[Cell 5] Final run on SV1...
[Cell 5] SV1 run failed (An error occurred (ValidationException) when calling the CreateQuantumTask operation: [line 516] result type state_vector is not supported on the requested device). Falling back to LocalSimulator for final decode.
[Cell 5] Expected QUBO cost (final): 534.567854
[Cell 5] Most probable bitstring:   001100011010001110  (p=6.656e-06, cost=-4.800000)
[Cell 5] Min-cost basis bitstring:  111100000000000000   (p=4.130e-06, cost=-642.400000)
[Cell 5] Exact global optimum:      111100000000000000 (cost=-642.400000)
[Cell 5] Min-cost bitstring selects 4 trials (within K=18).
[Cell 5] Total estimated cost: 8.00
[Cell 5] Total benefit score:  2.400000


,var_index,nct_id,phase,overall_status,estimated_trial_cost,benefit_score,lead_sponsor_norm,region_label
0,0,NCT00003042,Phase 2,"Active, not recruiting",2.0,0.6,City of Hope Medical Center,Global / Multi-Region
1,1,NCT06234631,Phase 2,Recruiting,2.0,0.6,Chad Brummett,Global / Multi-Region
2,2,NCT06257537,Phase 2,Recruiting,2.0,0.6,"ZetrOZ, Inc.",Global / Multi-Region
3,3,NCT06257875,Phase 2,"Active, not recruiting",2.0,0.6,AbbVie,Global / Multi-Region


### What Cell 5 Just Did

This cell took the best QAOA parameters found in Cell 4 and executed a single “final” statevector run (SV1 preferred, with a safe fallback to LocalSimulator if SV1 fails). Using the resulting quantum state, it computed the deterministic expected QUBO objective value and then decoded two concrete candidate solutions:

- the **most-probable** basis state under the final quantum state, and  
- the **minimum-cost** basis state under the QUBO (by scanning all basis states of the reduced K-variable problem).

When K is small enough, it also brute-forced the **exact global optimum** to provide a ground-truth baseline for the reduced subproblem. Finally, it mapped the selected bits (bit *i* → variable *i*) back to real trial rows via `meta_sub`, and summarized the total estimated cost and benefit for the selected trial set.


In [6]:
# ============================================================
# Cell 6 — Persist Phase 3 artifacts (small, trackable outputs)
# ============================================================

from pathlib import Path

RESULTS_DIR = Path("data/results")
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

bitstring_path = RESULTS_DIR / "03d_sv1_min_cost_bitstring_qubit0_first.txt"
summary_path = RESULTS_DIR / "03d_sv1_summary.csv"
selected_trials_path = RESULTS_DIR / "03d_sv1_selected_trials.csv"

# Save min-cost bitstring (explicitly qubit0-first)
with bitstring_path.open("w") as f:
    f.write(decoded["min_cost_bitstring"] + "\n")

summary = pd.DataFrame([{
    "notebook": "03d_qaoa_sv1_run",
    "device_final": "SV1" if FINAL_RUN_ON_SV1 else "LocalSimulator",
    "optimized_on": "LocalSimulator" if OPTIMIZE_LOCALLY else "SV1",
    "p": int(p),
    "K_variables": int(N),
    "expected_qubo_cost": float(exp_cost_final),

    "most_prob_bitstring": decoded["most_prob_bitstring"],
    "most_prob_probability": float(decoded["most_prob_probability"]),
    "most_prob_cost": float(decoded["most_prob_cost"]),

    "min_cost_bitstring": decoded["min_cost_bitstring"],
    "min_cost_probability": float(decoded["min_cost_probability"]),
    "min_cost_cost": float(decoded["min_cost_cost"]),

    "exact_global_opt_bitstring": global_opt_bitstring if global_opt_bitstring is not None else "",
    "exact_global_opt_cost": float(global_opt_cost) if global_opt_cost is not None else np.nan,

    "selected_count": int(len(selected_trials)),
    "selected_total_estimated_cost": float(total_cost),
    "selected_total_benefit": float(total_benefit),
}])

summary.to_csv(summary_path, index=False)
selected_trials.to_csv(selected_trials_path, index=False)

log(f"[Cell 6] Wrote {bitstring_path}")
log(f"[Cell 6] Wrote {summary_path}")
log(f"[Cell 6] Wrote {selected_trials_path}")

[Cell 6] Wrote data/results/03d_sv1_min_cost_bitstring_qubit0_first.txt
[Cell 6] Wrote data/results/03d_sv1_summary.csv
[Cell 6] Wrote data/results/03d_sv1_selected_trials.csv


### What Cell 6 Just Did

This cell saved a compact set of Phase 3 outputs under `data/results/` that are suitable for Git tracking and downstream reporting:

- A text file containing the selected **min-cost** bitstring, explicitly labeled **qubit0-first** to avoid ambiguity.
- A one-row CSV summary capturing QAOA settings (K, p), execution mode (local vs SV1), expected objective value, the decoded candidate bitstrings, and (when feasible) the brute-force optimum baseline.
- A CSV listing the selected trial rows mapped back from the bitstring via `meta_sub`.

These artifacts make the SV1 run reproducible and easy to reference in later notebooks, plots, and slides.


## Notebook Summary — Phase 3 QAOA (SV1)

In this notebook we:

1. Loaded the Scenario B QUBO and metadata and reduced it to a manageable K-variable subproblem.
2. Converted the QUBO to an Ising formulation (h, J) so the cost unitary can be built from native quantum gates.
3. Constructed QAOA circuits and evaluated the objective deterministically via statevector expectation values.
4. Optimized QAOA parameters (preferably locally for speed/cost), then ran a final confirmation job (SV1 preferred).
5. Decoded concrete candidate selections and compared them to an exact brute-force baseline when K is small enough.
6. Persisted small, trackable outputs to `data/results/`.

Next steps:
- Increase QAOA depth (p = 2, 3) and compare objective value, solution overlap, and selection stability.
- Add a sweep notebook (e.g., `03e_qaoa_sweep_p_and_K.ipynb`) to record results across (p, K, iterations) into a tidy table for plots.
- Increment K stepwise (e.g., 18 → 20 → 22) and track how performance and runtime scale.
